In [1]:
from helper_functions import LSE, construct_R, decomp_orthog
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

In [3]:
rng = np.random.default_rng(seed=100)

In [5]:
#This function, given an NxI np-array Mat, creates a new NXM np array Mat_twoway which contains all main-effects, quadratic terms, and
#two-way interactions of the columns in Mat
def two_way_interaction(Mat):

    #Retrieve the number of columns in Mat
    num_cols_Mat = np.shape(Mat)[1]
    
    Mat_twoway = np.concatenate((Mat,Mat**2),axis = 1)
    for i in range(num_cols_Mat):
        for j in range(i+1, num_cols_Mat):
            interact_ij = np.array([Mat[:,i]*Mat[:,j]]).T
            Mat_twoway = np.concatenate((Mat_twoway,interact_ij), axis = 1)

    return Mat_twoway

In [7]:
#This function takes the scaling factors from decomp_orthog and multiplies them by the LSE estimates to get the scaled coefficients.
def decomp_scaled_coefficients(theta,lse_coeff):
    #Theta: this should be an MxJ numpy array of scaling factors obtained from decomp_orthog.
    #lse_coeff: this should be an MxJ numpy array of estimated coefficients coming from LSE, and should be the LSE estimates used in constructing matrices R_1,...
    #R_J for decomp_orthog.

    return theta*lse_coeff

In [9]:
#Constructs a list to be used in decomp_orthog to indicate heredity relations. We assume a full quadratics-interactions effects model is being fit. We assume that
#there is no constant term in the model.
def heredity_list(I,J):
    #I - number of input variables
    #J - number of objective function
    
    inner_list = []

    #Main effects
    for i in range(I):
        inner_list.append([])

    #Quadratics
    for i in range(I):
        inner_list.append([i])

    #Interactions
    for i in range(I):
        for i_2 in range(i+1,I):
            inner_list.append([i,i_2])

    D_her = [inner_list for j in range(J)]

    return D_her

In [11]:
#Define functions and parameters used in problem.
gamma = 7.0 * 10**(-3)

lam = 1.3

#We assume input_vector will have 12 dimensions. There are 10 objective functions (f), these objective functions are
#made up of basis functions (g)

def g1(input_vector):
    return (input_vector[0] - 1)**2 + (input_vector[1] - 1)**2 + (input_vector[2] - 1)**2

def g2(input_vector):
    return (input_vector[0] - 1)**2 + (input_vector[1] + 1)**2 + (input_vector[2] + 1)**2

def g3(input_vector):
    return (input_vector[0] - 1)**2 + (input_vector[1] - 1)**2 + (input_vector[2] + 1)**2

def g4(input_vector):
    return (input_vector[3] + 1)**2 + (input_vector[4] + 1)**2 + (input_vector[5] + 1)**2

def g5(input_vector):
    return (input_vector[3] + 1)**2 + (input_vector[4] - 1)**2 + (input_vector[5] + 1)**2

def g6(input_vector):
    return (input_vector[3] + 1)**2 + (input_vector[4] + 1)**2 + (input_vector[5] - 1)**2

def g7(input_vector):
    return (input_vector[6] - 1)**2 + (input_vector[7] + 1)**2 

def g8(input_vector):
    return (input_vector[6] + 1)**2 + (input_vector[7] - 1)**2

def g9(input_vector):
    return np.sin(input_vector[8]) + np.sin(input_vector[9]) + np.sin(input_vector[10]) + np.sin(input_vector[11]) + np.cos(input_vector[8]) + np.cos(input_vector[9]) + np.cos(input_vector[10]) + np.cos(input_vector[11])

def g10(input_vector):
    return np.sin(-input_vector[8]) + np.sin(-input_vector[9]) + np.sin(-input_vector[10]) + np.sin(-input_vector[11]) + np.cos(-input_vector[8]) + np.cos(-input_vector[9]) + np.cos(-input_vector[10]) + np.cos(-input_vector[11])

#Define the f functions

def f1(input_vector):
    return g1(input_vector) + gamma*g4(input_vector)

def f2(input_vector):
    return g2(input_vector) + gamma*g5(input_vector)

def f3(input_vector):
    return g3(input_vector) + gamma*(g4(input_vector) + g6(input_vector))

def f4(input_vector):
    return g4(input_vector) + gamma*(g1(input_vector) + g7(input_vector))

def f5(input_vector):
    return g5(input_vector) + gamma*(g2(input_vector) + g8(input_vector))

def f6(input_vector):
    return g6(input_vector) + gamma*(g3(input_vector) + g9(input_vector))

def f7(input_vector):
    return g7(input_vector) + gamma*(g2(input_vector) + g5(input_vector))

def f8(input_vector):
    return g8(input_vector) + gamma*(g1(input_vector) + g4(input_vector) + g9(input_vector))

def f9(input_vector):
    return g9(input_vector) + gamma*(g3(input_vector) + g6(input_vector) + g8(input_vector))

def f10(input_vector):
    return g10(input_vector) + gamma*(g4(input_vector) + g5(input_vector))

def f_vec(input_vector):
    return [f1(input_vector),f2(input_vector),f3(input_vector),f4(input_vector),f5(input_vector),f6(input_vector),f7(input_vector),f8(input_vector),f9(input_vector),f10(input_vector)]

In [13]:
#model fitting process. set things up for decomposition.
num_dim_design = 12
num_dim_objective = 10

#Generate 104 points randomly from [-lam,lam]^12. (like in the Tabatabaei paper)
sample_size_fit = 104
design_matrix = rng.uniform(-lam, lam, (sample_size_fit,num_dim_design))
model_matrix = two_way_interaction(design_matrix)
model_matrix_with_intercept = np.concatenate((np.ones((sample_size_fit,1)),model_matrix),axis = 1)

response_matrix = np.zeros((sample_size_fit,num_dim_objective))

for i in range(sample_size_fit):
    f_vals = f_vec(design_matrix[i])
    for j in range(num_dim_objective):
        response_matrix[i,j] = f_vals[j]

#Fit responses with intercepts
R1,LSE1 = construct_R(model_matrix_with_intercept,response_matrix[:,0].reshape(sample_size_fit,))
R2,LSE2 = construct_R(model_matrix_with_intercept,response_matrix[:,1].reshape(sample_size_fit,))
R3,LSE3 = construct_R(model_matrix_with_intercept,response_matrix[:,2].reshape(sample_size_fit,))
R4,LSE4 = construct_R(model_matrix_with_intercept,response_matrix[:,3].reshape(sample_size_fit,))
R5,LSE5 = construct_R(model_matrix_with_intercept,response_matrix[:,4].reshape(sample_size_fit,))
R6,LSE6 = construct_R(model_matrix_with_intercept,response_matrix[:,5].reshape(sample_size_fit,))
R7,LSE7 = construct_R(model_matrix_with_intercept,response_matrix[:,6].reshape(sample_size_fit,))
R8,LSE8 = construct_R(model_matrix_with_intercept,response_matrix[:,7].reshape(sample_size_fit,))
R9,LSE9 = construct_R(model_matrix_with_intercept,response_matrix[:,8].reshape(sample_size_fit,))
R10,LSE10 = construct_R(model_matrix_with_intercept,response_matrix[:,9].reshape(sample_size_fit,))

#Remove the intercept term from the response
Y_1 = response_matrix[:,0].reshape(sample_size_fit,) - LSE1[0]
Y_2 = response_matrix[:,1].reshape(sample_size_fit,) - LSE2[0]
Y_3 = response_matrix[:,2].reshape(sample_size_fit,) - LSE3[0]
Y_4 = response_matrix[:,3].reshape(sample_size_fit,) - LSE4[0]
Y_5 = response_matrix[:,4].reshape(sample_size_fit,) - LSE5[0]
Y_6 = response_matrix[:,5].reshape(sample_size_fit,) - LSE6[0]
Y_7 = response_matrix[:,6].reshape(sample_size_fit,) - LSE7[0]
Y_8 = response_matrix[:,7].reshape(sample_size_fit,) - LSE8[0]
Y_9 = response_matrix[:,8].reshape(sample_size_fit,) - LSE9[0]
Y_10 = response_matrix[:,9].reshape(sample_size_fit,) - LSE10[0]

#Refit without intercept term
R1_noint,LSE1_noint = construct_R(model_matrix,Y_1)
R2_noint,LSE2_noint = construct_R(model_matrix,Y_2)
R3_noint,LSE3_noint = construct_R(model_matrix,Y_3)
R4_noint,LSE4_noint = construct_R(model_matrix,Y_4)
R5_noint,LSE5_noint = construct_R(model_matrix,Y_5)
R6_noint,LSE6_noint = construct_R(model_matrix,Y_6)
R7_noint,LSE7_noint = construct_R(model_matrix,Y_7)
R8_noint,LSE8_noint = construct_R(model_matrix,Y_8)
R9_noint,LSE9_noint = construct_R(model_matrix,Y_9)
R10_noint,LSE10_noint = construct_R(model_matrix,Y_10)

#Recall we are fitting the full quadratic model for each objective.
heredity_list = heredity_list(num_dim_design,num_dim_objective)

#We will use these in the decomposition process below.
responses = np.array([Y_1,Y_2,Y_3,Y_4,Y_5,Y_6,Y_7,Y_8,Y_9,Y_10])

In [15]:
#save design matrix.
design_df = pd.DataFrame(design_matrix, columns=['x'+str(i+1) for i in range(0,num_dim_design)])
design_df.to_csv('design_matrix_highdim_fit.csv', index=False)

In [ ]:
#Begin the decomposition procedure
decomp_fit = decomp_orthog(responses,[R1_noint,R2_noint,R3_noint,R4_noint,R5_noint,R6_noint,R7_noint,R8_noint,R9_noint,R10_noint],heredity,4,t=300,focus=0,outputflag = 1)
#print(decomp_fit)

#Get the scaled coefficients
decomp_coeff = decomp_scaled_coefficients(decomp_fit[0].T,np.array([LSE1_noint,LSE2_noint,LSE3_noint,LSE4_noint,LSE5_noint,LSE6_noint,LSE7_noint,LSE8_noint,LSE9_noint,LSE10_noint]).T)

In [19]:
main_effect_list = ['x'+str(i+1) for i in range(0,num_dim_design)]
quadratic_effect_list = ['x'+str(i+1)+'**2' for i in range(0,num_dim_design)]
interaction_effect_list = ['x'+str(i+1)+'*'+'x'+str(j+1) for i in range(0,num_dim_design - 1) for j in range(i+1,num_dim_design)]
all_effect_list = main_effect_list + quadratic_effect_list + interaction_effect_list
#decomp_coeff = rng.uniform(-1,1,(len(all_effect_list),num_dim_objective))
decomp_coeff_df = pd.DataFrame(decomp_coeff, columns = ['f'+str(i+1) for i in range(0,num_dim_objective)])
decomp_coeff_df.insert(0, 'Effect', all_effect_list)
decomp_coeff_df.to_csv('decomp_coeff_highdim.csv', index = False)